# Fase 4 — Clasificación neuronal con sklearn MLPClassifier
## MLP (Multilayer Perceptron) sobre vectores TF-IDF

**Proyecto:** Clasificador Masivo de Noticias · **Equipo:** DataWhales  
**Curso:** Datos Masivos I — Licenciatura en Ciencia de Datos, UNAM, 2026-2

---

**Arquitectura MLP real:**
```
Entrada (4096) → Oculta (128, ReLU) → Salida (18 clases, Softmax)
Optimizador: Adam  |  max_iter: 30  |  Parámetros totales: 526,738
```

**Entrada:** `datos/processed/tfidf_mind/` — vectores TF-IDF de MIND Large (Fase 2)  
**Salida:** `models/mlp_sklearn.joblib` + `datos/processed/predicciones/`

## 0. Rutas

In [1]:
import os

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")
MODELS_PATH  = os.path.join(PROJECT_PATH, "models")

TFIDF_MIND   = os.path.join(DATOS_PATH,  "processed", "tfidf_mind")
OUT_MODEL    = os.path.join(MODELS_PATH, "mlp_sklearn.joblib")
OUT_PREDS    = os.path.join(DATOS_PATH,  "processed", "predicciones")

os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(OUT_PREDS,   exist_ok=True)

print(f"PROJECT_PATH -> {PROJECT_PATH}")
print(f"TFIDF_MIND   -> {TFIDF_MIND}  ({'OK' if os.path.exists(TFIDF_MIND) else 'FALTA'})")

PROJECT_PATH -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos
TFIDF_MIND   -> /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/tfidf_mind  (OK)


## 1. Dependencias

In [2]:
# Ejecuta solo la primera vez
# !pip install pyspark scikit-learn scipy joblib

## 2. Inicialización de PySpark

PySpark se usa en esta fase exclusivamente para:
- Leer los vectores TF-IDF desde Parquet
- Codificar etiquetas con `StringIndexer`
- Reducir dimensionalidad con `ChiSqSelector`
- Dividir train/test de forma reproducible
- Guardar predicciones en Parquet

El entrenamiento del MLP **no ocurre en Spark** — ver sección 6.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Clasificador-MLP")
    .master("local[2]")
    .config("spark.driver.memory", "10g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

26/06/06 19:51:15 WARN Utils: Your hostname, MacBook-Pro-de-Milena.local resolves to a loopback address: 127.0.0.1; using 192.168.1.66 instead (on interface en0)
26/06/06 19:51:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 19:51:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/06 19:51:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


PySpark 3.5.1 listo.


## 3. Carga de datos (salida de Fase 2)

In [4]:
df = spark.read.parquet(TFIDF_MIND)

print(f"Total documentos: {df.count():,}")
print(f"Columnas: {df.columns}")

print("\nDistribución de categorías:")
df.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

Total documentos: 172,422
Columnas: ['doc_id', 'label', 'tfidf_vector']

Distribución de categorías:


+-------------+-----+
|label        |count|
+-------------+-----+
|sports       |53112|
|news         |52189|
|finance      |10217|
|travel       |8291 |
|lifestyle    |7975 |
|foodanddrink |7855 |
|video        |7536 |
|weather      |6972 |
|autos        |5358 |
|health       |5302 |
|tv           |2390 |
|music        |2172 |
|entertainment|1549 |
|movies       |1383 |
|kids         |114  |
|middleeast   |4    |
|games        |2    |
|northamerica |1    |
+-------------+-----+



## 4. Codificación de etiquetas

`StringIndexer` de PySpark MLlib convierte las 18 categorías de texto a índices numéricos
de forma determinista (ordena por frecuencia descendente). Esto es necesario porque
sklearn MLPClassifier requiere etiquetas enteras.

In [5]:
from pyspark.ml.feature import StringIndexer

indexer       = StringIndexer(inputCol="label", outputCol="label_idx", handleInvalid="skip")
indexer_model = indexer.fit(df)
df_indexed    = indexer_model.transform(df)

labels   = indexer_model.labels
N_CLASES = len(labels)

print(f"Clases ({N_CLASES}):")
for i, l in enumerate(labels):
    print(f"  {i:2d} -> {l}")

26/06/06 19:51:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Clases (18):
   0 -> sports
   1 -> news
   2 -> finance
   3 -> travel
   4 -> lifestyle
   5 -> foodanddrink
   6 -> video
   7 -> weather
   8 -> autos
   9 -> health
  10 -> tv
  11 -> music
  12 -> entertainment
  13 -> movies
  14 -> kids
  15 -> middleeast
  16 -> games
  17 -> northamerica


## 5. Reducción de dimensionalidad con ChiSqSelector (PySpark MLlib)

Los vectores TF-IDF tienen 65,536 dimensiones (2^16). Con esa dimensionalidad,
PySpark MLlib MLP genera bloques densos de ~33 MB por partición en la JVM → OOM en local.

`ChiSqSelector` selecciona las 4,096 features con mayor correlación estadística
(test chi-cuadrado) con la etiqueta de categoría. Esto reduce el bloque a ~2 MB
y hace viable el entrenamiento en local.

**Nota:** Esta operación **sí corre en Spark** distribuido sobre las 172,422 filas.

In [6]:
from pyspark.ml.feature import ChiSqSelector

# Reduccion de dimensionalidad: 65536 → 4096 features
# Con 65536 dims el MLP genera bloques de 33 MB/particion → OOM en local.
# ChiSqSelector selecciona las 4096 features con mayor correlacion con la etiqueta.
# Bloque resultante: 128 × 4096 × 4 bytes = 2 MB → cabe en memoria.
TOP_FEATURES = 4096

selector = ChiSqSelector(
    numTopFeatures=TOP_FEATURES,
    featuresCol="tfidf_vector",
    outputCol="selected_features",
    labelCol="label_idx"
)

print(f"Ajustando ChiSqSelector: 65536 → {TOP_FEATURES} features...")
selector_model = selector.fit(df_indexed)
df_selected = selector_model.transform(df_indexed)

print(f"Reducción completada. Schema:")
df_selected.select("doc_id", "label_idx", "selected_features").limit(2).show(truncate=60)

Ajustando ChiSqSelector: 65536 → 4096 features...


Reducción completada. Schema:
+------+---------+------------------------------------------------------------+
|doc_id|label_idx|                                           selected_features|
+------+---------+------------------------------------------------------------+
|N93570|      1.0|(4096,[301,921,1327,2326,3424,3695,3910],[5.7138256048598...|
|N80418|      1.0|(4096,[138,430,1604,1639,1818,2615,3098,3483,3766],[3.006...|
+------+---------+------------------------------------------------------------+



## 5b. Split train/test (PySpark)

División aleatoria 80/20 reproducible con `seed=42`.

In [7]:
if "split" in df_selected.columns:
    train_df = df_selected.filter(F.col("split") == "train")
    test_df  = df_selected.filter(F.col("split") == "dev")
    print("Split por campo 'split' original de MIND:")
else:
    train_df, test_df = df_selected.randomSplit([0.8, 0.2], seed=42)
    print("Split aleatorio 80/20:")

print(f"  Train: {train_df.count():,} docs")
print(f"  Test:  {test_df.count():,} docs")

Split aleatorio 80/20:


  Train: 137,887 docs


  Test:  34,535 docs


## 6. Entrenamiento del MLP con sklearn MLPClassifier

**¿Por qué sklearn y no PySpark MLlib?**

PySpark MLlib `MultilayerPerceptronClassifier` requiere materializar bloques
densos en la JVM. Con vectores de 65,536 dimensiones, eso produce bloques de
~33 MB por partición → `OutOfMemoryError` en una laptop con 10 GB asignados al driver.

sklearn `MLPClassifier` acepta directamente matrices `scipy.sparse.csr_matrix`,
que nunca se densifican en memoria. Esto hace viable el entrenamiento en local.

**Flujo híbrido Spark + sklearn:**
1. Spark extrae una muestra de 20k train / 5k test como matrices scipy sparse
2. sklearn entrena el MLP sobre esa muestra (sin JVM)
3. Las predicciones regresan a Spark para guardarse en Parquet

**Arquitectura real entrenada:**
```
Entrada (4096) → Oculta (128, ReLU) → Salida (18, Softmax)
Optimizador: Adam  |  max_iter: 30  |  Parámetros: 526,738
```

**Nota sobre convergencia:** sklearn emite `ConvergenceWarning` porque el modelo
no convergió en 30 iteraciones. Esto se debe a la limitación de cómputo en local;
en producción se aumentaría `max_iter` o se usaría early stopping.

In [8]:
import time
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neural_network import MLPClassifier

# Muestra extraida de Spark para entrenar/evaluar sklearn
# El corpus completo (137k train) tarda horas en local con MLP;
# 20k es suficiente para demostrar el pipeline.
TRAIN_SAMPLE = 20_000
TEST_SAMPLE  =  5_000

def spark_to_scipy(spark_df, features_col, label_col, n_sample):
    """Convierte columna SparseVector de Spark a scipy.sparse.csr_matrix."""
    pdf = spark_df.select(features_col, label_col).limit(n_sample).toPandas()
    rows, cols, vals = [], [], []
    for i, vec in enumerate(pdf[features_col]):
        for j, v in zip(vec.indices, vec.values):
            rows.append(i); cols.append(int(j)); vals.append(float(v))
    X = csr_matrix((vals, (rows, cols)), shape=(len(pdf), TOP_FEATURES))
    y = pdf[label_col].astype(int).values
    return X, y

print("Extrayendo muestra train/test de Spark → scipy sparse...")
X_train, y_train = spark_to_scipy(train_df, "selected_features", "label_idx", TRAIN_SAMPLE)
X_test,  y_test  = spark_to_scipy(test_df,  "selected_features", "label_idx", TEST_SAMPLE)
print(f"Train: {X_train.shape}  nnz={X_train.nnz:,}")
print(f"Test:  {X_test.shape}")

capas = [TOP_FEATURES, 128, N_CLASES]  # [4096, 128, 18]

mlp_sk = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    max_iter=30,
    random_state=42,
    verbose=True)

print(f"\nArquitectura MLP (sklearn): {capas}")
# print("Entrenando sin JVM — estimado 3-8 min...")
t0 = time.time()
mlp_sk.fit(X_train, y_train)
print(f"\nEntrenamiento completado en {(time.time()-t0)/60:.1f} min")

Extrayendo muestra train/test de Spark → scipy sparse...


Train: (20000, 4096)  nnz=205,295
Test:  (5000, 4096)

Arquitectura MLP (sklearn): [4096, 128, 18]
Iteration 1, loss = 1.51026908
Iteration 2, loss = 0.74950899
Iteration 3, loss = 0.48497251
Iteration 4, loss = 0.33483262
Iteration 5, loss = 0.23787894
Iteration 6, loss = 0.17410979
Iteration 7, loss = 0.13041636
Iteration 8, loss = 0.10040489
Iteration 9, loss = 0.08047058
Iteration 10, loss = 0.06611251
Iteration 11, loss = 0.05455088
Iteration 12, loss = 0.04678539
Iteration 13, loss = 0.04065279
Iteration 14, loss = 0.03671349
Iteration 15, loss = 0.03306948
Iteration 16, loss = 0.03052888
Iteration 17, loss = 0.02722727
Iteration 18, loss = 0.02691541
Iteration 19, loss = 0.02484131
Iteration 20, loss = 0.02274185
Iteration 21, loss = 0.02176040
Iteration 22, loss = 0.02076762
Iteration 23, loss = 0.02050518
Iteration 24, loss = 0.01898666
Iteration 25, loss = 0.01999074
Iteration 26, loss = 0.01821569
Iteration 27, loss = 0.01721911
Iteration 28, loss = 0.01755662
Iteration 29, 

/Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


## 7. Predicciones y ejemplos

In [9]:
y_pred_train = mlp_sk.predict(X_train)
y_pred_test  = mlp_sk.predict(X_test)

print("Ejemplos de predicción (test):")
print(f"{'Real':<20} {'Predicho'}")
print("-" * 40)
for i in range(8):
    real = labels[y_test[i]]
    pred = labels[y_pred_test[i]]
    ok   = "OK" if real == pred else "X"
    print(f"{real:<20} {pred}  {ok}")

Ejemplos de predicción (test):
Real                 Predicho
----------------------------------------
weather              weather  OK
travel               travel  OK
sports               sports  OK
sports               sports  OK
sports               sports  OK
news                 movies  X
finance              sports  X
news                 news  OK


## 8. Métricas de evaluación

In [10]:
from sklearn.metrics import accuracy_score, f1_score

acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test,  y_pred_test)
f1_test   = f1_score(y_test, y_pred_test, average="weighted", zero_division=0)

print("=" * 40)
print("RESULTADOS DEL CLASIFICADOR MLP")
print("=" * 40)
print(f"  Accuracy train : {acc_train:.4f}")
print(f"  Accuracy test  : {acc_test:.4f}")
print(f"  F1 ponderado   : {f1_test:.4f}")
print("=" * 40)

RESULTADOS DEL CLASIFICADOR MLP
  Accuracy train : 0.9964
  Accuracy test  : 0.7560
  F1 ponderado   : 0.7543


In [11]:
from sklearn.metrics import classification_report

present = sorted(set(y_test) | set(y_pred_test))
present_names = [labels[i] for i in present]

print(classification_report(
    y_test, y_pred_test,
    labels=present,
    target_names=present_names,
    zero_division=0
))

               precision    recall  f1-score   support

       sports       0.89      0.91      0.90      1550
         news       0.76      0.79      0.78      1542
      finance       0.54      0.64      0.59       279
       travel       0.56      0.55      0.56       237
    lifestyle       0.55      0.52      0.54       231
 foodanddrink       0.76      0.69      0.72       223
        video       0.57      0.52      0.54       210
      weather       0.75      0.74      0.74       224
        autos       0.71      0.65      0.68       165
       health       0.60      0.51      0.55       141
           tv       0.71      0.52      0.60        58
        music       0.85      0.60      0.70        67
entertainment       0.63      0.58      0.60        38
       movies       0.70      0.56      0.62        34
         kids       0.00      0.00      0.00         1

     accuracy                           0.76      5000
    macro avg       0.64      0.58      0.61      5000
 weighte

## 9. Guardado del modelo y predicciones

In [12]:
import joblib, os, pandas as pd

# Guardar modelo sklearn con joblib (formato nativo de sklearn)
joblib.dump(mlp_sk, os.path.join(MODELS_PATH, "mlp_sklearn.joblib"))
print(f"Modelo guardado: {MODELS_PATH}/mlp_sklearn.joblib")

# Reconstruir predicciones con doc_id y etiquetas para Fase 5
test_meta = (test_df
    .select("doc_id", "label", "label_idx")
    .limit(TEST_SAMPLE)
    .toPandas())

test_meta["prediccion_idx"] = y_pred_test
test_meta["prediccion"]     = [labels[i] for i in y_pred_test]

# Las predicciones regresan a Spark para guardarse en Parquet
preds_spark = spark.createDataFrame(test_meta)
(preds_spark
    .write.mode("overwrite")
    .parquet(OUT_PREDS))
print(f"Predicciones guardadas: {OUT_PREDS}")

Modelo guardado: /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/models/mlp_sklearn.joblib


Predicciones guardadas: /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/predicciones


## 10. Modelo costo-comunicación — Fase 4

El análisis de costo refleja el pipeline híbrido **real** implementado,
no una arquitectura distribuida hipotética.

| Etapa | Dónde corre | Costo computacional | Comunicación |
|---|---|---|---|
| Leer Parquet TF-IDF | Spark local | O(N) | O(N) — lectura de disco |
| StringIndexer | Spark local | O(N) | O(1) — broadcast del modelo |
| ChiSqSelector (fit) | Spark local | O(N × \|V\|) | O(\|V\|) — shuffle de conteos |
| ChiSqSelector (transform) | Spark local | O(N × TOP_K) | O(N × TOP_K) |
| spark_to_scipy (collect) | Driver | O(muestra) | O(muestra) — collect al driver |
| MLP fit (sklearn) | Driver, 1 proceso | O(iter × muestra × params) | 0 — sin red |
| Guardar predicciones | Spark local | O(muestra) | O(muestra) — escritura Parquet |

**Parámetros del modelo entrenado:**

In [13]:
n_params = sum(capas[i]*capas[i+1]+capas[i+1] for i in range(len(capas)-1))

print("MODELO COSTO-COMUNICACIÓN — CLASIFICADOR MLP (Fase 4)")
print("-" * 55)
print(f"Librería de entrenamiento  : sklearn MLPClassifier")
print(f"Documentos train (muestra) : {TRAIN_SAMPLE:,}  (de {train_df.count():,} disponibles)")
print(f"Documentos test  (muestra) : {TEST_SAMPLE:,}   (de {test_df.count():,} disponibles)")
print(f"Dimensiones entrada        : {TOP_FEATURES:,}   (reducidas de 65,536 con ChiSqSelector)")
print(f"Arquitectura               : {capas}")
print(f"Parámetros totales         : {n_params:,}")
print(f"Optimizador                : Adam")
print(f"Iteraciones                : 30 (sin convergencia — ConvergenceWarning)")
print()
print(f"Cuello de botella real     : collect() de Spark al driver para spark_to_scipy")
print(f"Costo collect              : O(muestra × TOP_K) = O(20000 × 4096) ~ 328 MB")
print()
print(f"Accuracy test              : {acc_test:.4f}")
print(f"F1 ponderado               : {f1_test:.4f}")

MODELO COSTO-COMUNICACIÓN — CLASIFICADOR MLP (Fase 4)
-------------------------------------------------------
Librería de entrenamiento  : sklearn MLPClassifier


Documentos train (muestra) : 20,000  (de 137,887 disponibles)


Documentos test  (muestra) : 5,000   (de 34,535 disponibles)
Dimensiones entrada        : 4,096   (reducidas de 65,536 con ChiSqSelector)
Arquitectura               : [4096, 128, 18]
Parámetros totales         : 526,738
Optimizador                : Adam
Iteraciones                : 30 (sin convergencia — ConvergenceWarning)

Cuello de botella real     : collect() de Spark al driver para spark_to_scipy
Costo collect              : O(muestra × TOP_K) = O(20000 × 4096) ~ 328 MB

Accuracy test              : 0.7560
F1 ponderado               : 0.7543


In [14]:
spark.stop()
print("SparkSession cerrada. Fase 4 completada.")
print("Siguiente paso: 05_evaluacion.ipynb")

SparkSession cerrada. Fase 4 completada.
Siguiente paso: 05_evaluacion.ipynb
